<a href="https://colab.research.google.com/github/nkefeyan-22-26/ECON5200-Applied-Data-Analytics-in-Economics/blob/main/MidtermProject/data/raw/notebooks/02_Replication_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nathan Kefeyan
# ECON5200 Midterm Project
# Instrumental Variables Recreation - Angrist & Evans 1998 Recreation
Phase 2: Replication Analysis

In [2]:
import pandas as pd
import numpy as np

In [19]:
# Reading the FULL file: it contains many columns that likely won't be used in any analysis
FILEPATH = '/content/census80_full.csv.gz'
df_full = pd.read_csv(FILEPATH)
df_full.head()

,STATE,SEXK,AGEK,QTRBKID,RACEK,SPANISH,BIRTHPLK,SCHOOLK,GRADE,FINGRADE,...,FINGRADD,CLASSD,WEEKSD,HOURSD,INCOME1D,INCOME2D,AWEEK79D,AHOUR79D,AINC1D,AINC2D
0,b'01',b'1',b'09',b'3',b'01',b'0',b'001',b'1',b'06',b'1',...,b'2',b'1',b'52',b'40',b'28005',b'00000',b'0',b'0',b'0',b'0'
1,b'01',b'1',b'08',b'3',b'01',b'0',b'001',b'2',b'05',b'1',...,b'2',b'5',b'52',b'72',b'00000',b'10005',b'0',b'0',b'0',b'3'
2,b'01',b'0',b'09',b'1',b'01',b'0',b'001',b'1',b'05',b'1',...,b'2',b'5',b'16',b'48',b'00000',b'16005',b'0',b'0',b'2',b'0'
3,b'01',b'0',b'05',b'2',b'01',b'0',b'001',b'1',b'02',b'1',...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,b'01',b'1',b'11',b'1',b'01',b'0',b'001',b'1',b'07',b'1',...,b'2',b'1',b'32',b'40',b'09925',b'00000',b'0',b'0',b'3',b'3'


In [3]:
# Reading the SLIM file: it columns that are used in the paper
FILEPATH= '/content/census80_slim.csv.gz'
df_slim = pd.read_csv(FILEPATH)
df_slim.columns = df_slim.columns.str.lower()
df_slim.head()

,sexk,sex2nd,kidcount,agem,racem,spanishm,gradem,marital,weeksm,hoursm,income1m,income2m,aged,raced,graded,weeksd,hoursd,income1d,faminc,state
0,b'1',b'0',2.0,b'36',b'01',b'0',b'14',b'0',b'16',b'30',b'02405',b'00000',b'39',b'01',b'14',b'52',b'40',b'28005',b'33385',b'01'
1,b'1',NaN,1.0,b'37',b'01',b'0',b'17',b'0',b'32',b'35',b'05005',b'00000',b'39',b'01',b'15',b'52',b'72',b'00000',b'16460',b'01'
2,b'0',b'1',2.0,b'27',b'01',b'0',b'14',b'0',b'00',b'00',b'00000',b'00000',b'35',b'01',b'14',b'16',b'48',b'00000',b'16005',b'01'
3,b'0',b'1',2.0,b'25',b'01',b'0',b'14',b'2',b'52',b'38',b'08705',b'00000',NaN,NaN,NaN,NaN,NaN,NaN,b'10310',b'01'
4,b'1',b'0',2.0,b'30',b'01',b'0',b'10',b'0',b'30',b'40',b'09005',b'00000',b'28',b'01',b'11',b'32',b'40',b'09925',b'20640',b'01'


In [4]:
def decode_col(col):
    if col.dtype == object:
        col = col.str.lstrip("b'").str.rstrip("'")
    return pd.to_numeric(col, errors='coerce')

df_slim = df_slim.apply(decode_col)
print(df_slim['agem'].head(10))


0    36
1    37
2    27
3    25
4    30
5    27
6    20
7    23
8    25
9    44
Name: agem, dtype: int64


In [6]:
# Construct Instrument and Key Variables
df_slim['samesex'] = (df_slim['sexk'] == df_slim['sex2nd']).astype(int)
df_slim['morekids'] = (df_slim['kidcount'] > 2).astype(int)

print(f"Same sex share: {df_slim['samesex'].mean():.3f}  (expect ~0.505)")
print(f"More kids share: {df_slim['morekids'].mean():.3f}")

Same sex share: 0.357  (expect ~0.505)
More kids share: 0.271


In [7]:
# Testing values
# print(df_slim['agem'].value_counts(dropna=False))
print(df_slim['weeksm'].value_counts(dropna=False))

weeksm
0     361824
52    225228
40     30681
50     29423
36     18248
12     17211
48     16900
20     16877
16     13140
30     12328
26     11182
8      10575
24      9340
32      9086
4       7898
10      7361
51      7038
44      6929
6       6787
42      6334
28      6313
25      6046
45      5818
2       5791
49      5596
46      5570
1       4916
3       4824
38      4735
35      4608
15      4030
14      3822
18      3788
5       3557
22      2912
39      2812
13      2656
47      2622
9       2242
34      2237
37      2168
17      2062
7       1940
43      1727
21      1526
41      1373
11      1315
27      1178
33      1108
23      1082
19       876
29       863
31       764
Name: count, dtype: int64


In [8]:
cols = ['morekids', 'samesex', 'agem', 'gradem', 'racem', 'weeksm']
df_slim_clean = df_slim[cols].replace([np.inf, -np.inf], np.nan).dropna() # dropping out NaN values (most important) and potential infinity values

In [9]:
# First stage regression
import statsmodels.api as sm

X = sm.add_constant(df_slim_clean[['samesex', 'agem', 'gradem', 'racem']])
y = df_slim_clean['morekids']

first_stage = sm.OLS(y, X).fit()
print(first_stage.summary())
print(f"\nF-statistic: {first_stage.fvalue:.2f}  (must be > 10)")

                            OLS Regression Results                            
Dep. Variable:               morekids   R-squared:                       0.094
Model:                            OLS   Adj. R-squared:                  0.094
Method:                 Least Squares   F-statistic:                 2.396e+04
Date:                Sun, 22 Mar 2026   Prob (F-statistic):               0.00
Time:                        18:37:38   Log-Likelihood:            -5.1810e+05
No. Observations:              927267   AIC:                         1.036e+06
Df Residuals:                  927262   BIC:                         1.036e+06
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1447      0.003     43.833      0.0

An F-statistic of 23,961 is way above the threshold of 10. `samesex` has a highly significant relationship with `morekids`. The coefficient on `samesex` is 0.202, meaning that mothers whose first two children are the same sex are about 20 percentage points more likely to have a third child.

This aligns with Angrist & Evans' argumen: same-sex sibling pairs lead to parents having a preference for another child. The regression is large and statistically significant, which shows that the instrument is relevant. The next 2SLS estimates will also be meaningful due to this.

In [10]:
# Second stage Manual 2SLS
df_slim_clean.loc[df_slim_clean.index, 'morekids_hat'] = first_stage.fittedvalues

cols = ['weeksm', 'morekids_hat', 'agem', 'gradem', 'racem']
df_stage2 = df_slim_clean[cols].replace([np.inf, -np.inf], np.nan).dropna() # dropping out NaN values (most important) and potential infinity values

X2 = sm.add_constant(df_stage2[['morekids_hat', 'agem', 'gradem', 'racem']])
y2 = df_stage2['weeksm']

second_stage = sm.OLS(y2, X2).fit()
print(second_stage.summary())
print("Note: coefficients correct, standard errors are wrong")

                            OLS Regression Results                            
Dep. Variable:                 weeksm   R-squared:                       0.023
Model:                            OLS   Adj. R-squared:                  0.023
Method:                 Least Squares   F-statistic:                     5534.
Date:                Sun, 22 Mar 2026   Prob (F-statistic):               0.00
Time:                        18:37:39   Log-Likelihood:            -4.1966e+06
No. Observations:              927267   AIC:                         8.393e+06
Df Residuals:                  927262   BIC:                         8.393e+06
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            4.2427      0.180     23.543   

The manual 2SLS results show that having more than two children causes a mother to work about 16.6 less weeks per year. This figure is large and statistically significant. By using same-sex sibling composition as an instrument, we can isolate only the variation in family size, which is random. This is an improvement from the selection bias that was present in the naive OLS. The positive coefficients on `agem` and `gradem` make sense: older and more educated mothers tend to work more. The standard errors are incorrect because the 2SLS underestimates them.

In [11]:
!pip install linearmodels --quiet
from linearmodels.iv import IV2SLS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 7.3 MB/s eta 0:00:00


In [12]:
df_slim_clean = df_slim_clean[['weeksm', 'morekids', 'samesex', 'agem', 'gradem', 'racem']].dropna()

iv = IV2SLS(dependent = df_slim_clean['weeksm'],
            exog = sm.add_constant(df_slim_clean[['agem', 'gradem', 'racem']]),
            endog = df_slim_clean['morekids'],
            instruments = df_slim_clean['samesex']
            ).fit(cov_type='robust')

print(iv.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                 weeksm   R-squared:                     -0.0045
Estimator:                    IV-2SLS   Adj. R-squared:                -0.0045
No. Observations:              927267   F-statistic:                 2.285e+04
Date:                Sun, Mar 22 2026   P-value (F-stat)                0.0000
Time:                        18:37:56   Distribution:                  chi2(4)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          4.2427     0.1773     23.932     0.0000      3.8952      4.5901
agem           0.5055     0.0052     97.533     0.00

This production 2SLS confirms the result we got from manually running it. Having more than 2 children causes a mother to work about 16.6 less weeks per year (highly statistically significant). The coefficient is the same as the one in the manual 2SLS. The difference is that the standard errors are now correct becauswe linearmodels uses robust standard errors to account for heteroskedasticity (residuals aren't consistent, data "spreads").

In [13]:
# Wald Estimator
EY_Z1 = df_slim_clean.loc[df_slim_clean['samesex'] == 1, 'weeksm'].mean()
EY_Z0 = df_slim_clean.loc[df_slim_clean['samesex'] == 0, 'weeksm'].mean()
ED_Z1 = df_slim_clean.loc[df_slim_clean['samesex'] == 1, 'morekids'].mean()
ED_Z0 = df_slim_clean.loc[df_slim_clean['samesex'] == 0, 'morekids'].mean()

wald = (EY_Z1 - EY_Z0) / (ED_Z1 - ED_Z0)
numerator = EY_Z1 - EY_Z0
denominator = ED_Z1 - ED_Z0

print(f"Numerator: {numerator:.4f}")
print(f"Denominator: {denominator:.4f}")
print(f"Wald Estimator: {wald:.4f}")

Numerator: -3.1650
Denominator: 0.2170
Wald Estimator: -14.5849


The Wald estimator gives us a raw instrumental variables estimate of -14.6. This means that among mohters whos family size was due to same-sex siblings (and their preference for an additional child), having the additional child reduces weeks worked by about 14.6 weeks per year. Numerator / Denominator is "raw difference in years worked between mothers with same-sex and different-sex in the first two children" / "difference in probabiliyt btween those two groups that they will have a third children".